# -> Multiple model training, Pipeline and column Transformer

In [64]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

In [65]:
df=sns.load_dataset('tips')
df.head()

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


In [66]:
# predict what is the time ? if any one is visiting -- is it linch or dinner? Time is target variable

In [67]:
df.time.unique()

['Dinner', 'Lunch']
Categories (2, object): ['Lunch', 'Dinner']

In [68]:
## EDA >> Subjective

# encoding, missing valuetratment , scaling >> automate 

In [69]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 244 entries, 0 to 243
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   total_bill  244 non-null    float64 
 1   tip         244 non-null    float64 
 2   sex         244 non-null    category
 3   smoker      244 non-null    category
 4   day         244 non-null    category
 5   time        244 non-null    category
 6   size        244 non-null    int64   
dtypes: category(4), float64(2), int64(1)
memory usage: 7.4 KB


In [70]:
df.isnull().sum()

total_bill    0
tip           0
sex           0
smoker        0
day           0
time          0
size          0
dtype: int64

In [71]:
# Since time is nominal variable, then we will us elevel encoder
from sklearn.preprocessing import LabelEncoder
encoder=LabelEncoder()
encoder

LabelEncoder()

In [72]:
df['time']=encoder.fit_transform(df['time'])
df['time']

0      0
1      0
2      0
3      0
4      0
      ..
239    0
240    0
241    0
242    0
243    0
Name: time, Length: 244, dtype: int64

In [73]:
df

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,0,2
1,10.34,1.66,Male,No,Sun,0,3
2,21.01,3.50,Male,No,Sun,0,3
3,23.68,3.31,Male,No,Sun,0,2
4,24.59,3.61,Female,No,Sun,0,4
...,...,...,...,...,...,...,...
239,29.03,5.92,Male,No,Sat,0,3
240,27.18,2.00,Female,Yes,Sat,0,2
241,22.67,2.00,Male,Yes,Sat,0,2
242,17.82,1.75,Male,No,Sat,0,2


In [74]:
df['time'].unique() # dinner is 1, lunch is 0

array([0, 1])

In [75]:
x=df.drop('time',axis=1)
y=df['time']

In [76]:
x

,total_bill,tip,sex,smoker,day,size
0,16.99,1.01,Female,No,Sun,2
1,10.34,1.66,Male,No,Sun,3
2,21.01,3.50,Male,No,Sun,3
3,23.68,3.31,Male,No,Sun,2
4,24.59,3.61,Female,No,Sun,4
...,...,...,...,...,...,...
239,29.03,5.92,Male,No,Sat,3
240,27.18,2.00,Female,Yes,Sat,2
241,22.67,2.00,Male,Yes,Sat,2
242,17.82,1.75,Male,No,Sat,2


In [77]:
y

0      0
1      0
2      0
3      0
4      0
      ..
239    0
240    0
241    0
242    0
243    0
Name: time, Length: 244, dtype: int64

In [78]:
from sklearn.model_selection import train_test_split
x_train,x_test, y_train, y_test=train_test_split(x,y,test_size=0.20, random_state=1)

In [79]:
# Handling the missing value
# data encoding
# feature scaling

In [80]:
from sklearn.impute import SimpleImputer # For missing value
from sklearn.preprocessing import OneHotEncoder  #forencodinger 
from sklearn.preprocessing import StandardScaler # For scaling

from sklearn.pipeline import Pipeline  # A sequence of data transformer 
from sklearn.compose import ColumnTransformer # groups all the pipeline steps for each of the columns

In [81]:
df.sample()

,total_bill,tip,sex,smoker,day,time,size
37,16.93,3.07,Female,No,Sat,0,3


In [82]:
cat_cols=['sex','smoker','day']
nums_cols=['total_bill','tip','size']

## Feature engineering automation using pipeline and column transformer

In [83]:
num_pipeline=Pipeline(steps=[('imputation',SimpleImputer(strategy='median')),
               ('scaling',StandardScaler())])

cat_pipeline=Pipeline(steps=[('imputation',SimpleImputer(strategy='most_frequent')),
                            ('encoding',OneHotEncoder())])

In [84]:
preprocessor=ColumnTransformer([('num_pipeline',num_pipeline, nums_cols),
                  ('cat_pipeline',cat_pipeline, cat_cols)])
preprocessor

ColumnTransformer(transformers=[('num_pipeline',
                                 Pipeline(steps=[('imputation',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaling',
                                                  StandardScaler())]),
                                 ['total_bill', 'tip', 'size']),
                                ('cat_pipeline',
                                 Pipeline(steps=[('imputation',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoding',
                                                  OneHotEncoder())]),
                                 ['sex', 'smoker', 'day'])])

In [85]:
x_train=preprocessor.fit_transform(x_train)
x_test=preprocessor.transform(x_test)

In [86]:
x_train

array([[-0.28611937, -1.47443803, -0.57766863, ...,  0.        ,
         1.        ,  0.        ],
       [ 0.02695905, -0.71612531,  1.47042924, ...,  0.        ,
         1.        ,  0.        ],
       [ 1.3716196 ,  1.19880579,  1.47042924, ...,  0.        ,
         1.        ,  0.        ],
       ...,
       [-0.23206267,  0.43283335, -0.57766863, ...,  0.        ,
         0.        ,  1.        ],
       [-1.06543688, -1.29060464, -0.57766863, ...,  1.        ,
         0.        ,  0.        ],
       [-0.29287646,  0.1034652 ,  0.44638031, ...,  1.        ,
         0.        ,  0.        ]])

In [87]:
x_test

array([[-1.85376383, -1.48209775, -1.60171757,  1.        ,  0.        ,
         0.        ,  1.        ,  0.        ,  1.        ,  0.        ,
         0.        ],
       [-0.08453291,  0.04984713, -0.57766863,  1.        ,  0.        ,
         1.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         1.        ],
       [ 0.79501474,  0.36389583,  0.44638031,  0.        ,  1.        ,
         0.        ,  1.        ,  0.        ,  1.        ,  0.        ,
         0.        ],
       [-0.59356688, -0.33313909, -0.57766863,  0.        ,  1.        ,
         1.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         1.        ],
       [ 0.18349826,  0.04984713, -0.57766863,  0.        ,  1.        ,
         1.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         1.        ],
       [-1.32783714, -1.14506988, -0.57766863,  0.        ,  1.        ,
         0.        ,  1.        ,  0.        ,  1.        ,  0.        ,
         0.   

In [88]:
# Now data is ready , Lets build the multiple models

In [89]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

In [90]:
models={"support vector classifier":SVC(),
        "DT classifier": DecisionTreeClassifier(),
        "Log reg":LogisticRegression()
}
models

{'support vector classifier': SVC(),
 'DT classifier': DecisionTreeClassifier(),
 'Log reg': LogisticRegression()}

In [91]:
from sklearn.metrics import accuracy_score

def model_train_eval(x_train,x_test, y_train, y_test, models):
    evaluation={}
    for i in range (len(models)):
        model=list(models.values())[i]
        model.fit(x_train, y_train)
        y_pred=model.predict(x_test)
        model_score=accuracy_score(y_test, y_pred)
        evaluation[list(models.keys())[i]]=model_score
    return evaluation

In [92]:
model_train_eval(x_train,x_test, y_train, y_test, models)

{'support vector classifier': 0.9183673469387755,
 'DT classifier': 0.8979591836734694,
 'Log reg': 0.9183673469387755}

# -> Random Forest classifier

In [97]:
from sklearn.ensemble import RandomForestClassifier
rf=RandomForestClassifier()
rf

RandomForestClassifier()

In [98]:
x_train

array([[-0.28611937, -1.47443803, -0.57766863, ...,  0.        ,
         1.        ,  0.        ],
       [ 0.02695905, -0.71612531,  1.47042924, ...,  0.        ,
         1.        ,  0.        ],
       [ 1.3716196 ,  1.19880579,  1.47042924, ...,  0.        ,
         1.        ,  0.        ],
       ...,
       [-0.23206267,  0.43283335, -0.57766863, ...,  0.        ,
         0.        ,  1.        ],
       [-1.06543688, -1.29060464, -0.57766863, ...,  1.        ,
         0.        ,  0.        ],
       [-0.29287646,  0.1034652 ,  0.44638031, ...,  1.        ,
         0.        ,  0.        ]])

In [99]:
x_test

array([[-1.85376383, -1.48209775, -1.60171757,  1.        ,  0.        ,
         0.        ,  1.        ,  0.        ,  1.        ,  0.        ,
         0.        ],
       [-0.08453291,  0.04984713, -0.57766863,  1.        ,  0.        ,
         1.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         1.        ],
       [ 0.79501474,  0.36389583,  0.44638031,  0.        ,  1.        ,
         0.        ,  1.        ,  0.        ,  1.        ,  0.        ,
         0.        ],
       [-0.59356688, -0.33313909, -0.57766863,  0.        ,  1.        ,
         1.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         1.        ],
       [ 0.18349826,  0.04984713, -0.57766863,  0.        ,  1.        ,
         1.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         1.        ],
       [-1.32783714, -1.14506988, -0.57766863,  0.        ,  1.        ,
         0.        ,  1.        ,  0.        ,  1.        ,  0.        ,
         0.   

In [101]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
params={'max_depth':[1,2,3,4,5,10,None],
       'n_estimators':[30,40,50,100,200,300],
       'criterion':['gini','entropy']}
params

{'max_depth': [1, 2, 3, 4, 5, 10, None],
 'n_estimators': [30, 40, 50, 100, 200, 300],
 'criterion': ['gini', 'entropy']}

In [102]:
clf=RandomizedSearchCV(rf, param_distributions=params, cv=5, verbose=3, scoring='accuracy', n_iter=10)
clf

RandomizedSearchCV(cv=5, estimator=RandomForestClassifier(),
                   param_distributions={'criterion': ['gini', 'entropy'],
                                        'max_depth': [1, 2, 3, 4, 5, 10, None],
                                        'n_estimators': [30, 40, 50, 100, 200,
                                                         300]},
                   scoring='accuracy', verbose=3)

In [103]:
clf.fit(x_train,y_train)

Fitting 5 folds for each of 10 candidates, totalling 50 fits
[CV 1/5] END criterion=gini, max_depth=4, n_estimators=50;, score=0.923 total time=   0.2s
[CV 2/5] END criterion=gini, max_depth=4, n_estimators=50;, score=0.974 total time=   0.1s
[CV 3/5] END criterion=gini, max_depth=4, n_estimators=50;, score=1.000 total time=   0.1s
[CV 4/5] END criterion=gini, max_depth=4, n_estimators=50;, score=1.000 total time=   0.1s
[CV 5/5] END criterion=gini, max_depth=4, n_estimators=50;, score=1.000 total time=   0.1s
[CV 1/5] END criterion=gini, max_depth=2, n_estimators=100;, score=0.923 total time=   0.3s
[CV 2/5] END criterion=gini, max_depth=2, n_estimators=100;, score=0.974 total time=   0.3s
[CV 3/5] END criterion=gini, max_depth=2, n_estimators=100;, score=1.000 total time=   0.4s
[CV 4/5] END criterion=gini, max_depth=2, n_estimators=100;, score=1.000 total time=   0.3s
[CV 5/5] END criterion=gini, max_depth=2, n_estimators=100;, score=1.000 total time=   0.4s
[CV 1/5] END criterion=g

RandomizedSearchCV(cv=5, estimator=RandomForestClassifier(),
                   param_distributions={'criterion': ['gini', 'entropy'],
                                        'max_depth': [1, 2, 3, 4, 5, 10, None],
                                        'n_estimators': [30, 40, 50, 100, 200,
                                                         300]},
                   scoring='accuracy', verbose=3)

In [104]:
clf.best_params_

{'n_estimators': 50, 'max_depth': 4, 'criterion': 'gini'}

In [105]:
clf.best_score_

np.float64(0.9794871794871796)

In [107]:
final_model=clf.best_estimator_
final_model.predict(x_test)

array([0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0,
       0, 0, 0, 0, 0])

In [108]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

In [109]:
models={"support vector classifier":SVC(),
        "DT classifier": DecisionTreeClassifier(),
        "Log reg":LogisticRegression(),
        "RF":RandomForestClassifier()
}
models

{'support vector classifier': SVC(),
 'DT classifier': DecisionTreeClassifier(),
 'Log reg': LogisticRegression(),
 'RF': RandomForestClassifier()}

In [110]:
from sklearn.metrics import accuracy_score

def model_train_eval(x_train,x_test, y_train, y_test, models):
    evaluation={}
    for i in range (len(models)):
        model=list(models.values())[i]
        model.fit(x_train, y_train)
        y_pred=model.predict(x_test)
        model_score=accuracy_score(y_test, y_pred)
        evaluation[list(models.keys())[i]]=model_score
    return evaluation

In [111]:
model_train_eval(x_train,x_test, y_train, y_test, models)

{'support vector classifier': 0.9183673469387755,
 'DT classifier': 0.9183673469387755,
 'Log reg': 0.9183673469387755,
 'RF': 0.8979591836734694}

In [122]:
# Use total_bill as target variable and make it a regressor problem
# Automate fe, taraing, testing and also random forest regressor with MLR and SVR

In [ ]:
# Multiple model

In [123]:
df.head()

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,0,2
1,10.34,1.66,Male,No,Sun,0,3
2,21.01,3.50,Male,No,Sun,0,3
3,23.68,3.31,Male,No,Sun,0,2
4,24.59,3.61,Female,No,Sun,0,4


In [124]:
# Predict total_bill >>> target variable

In [125]:
x=df.drop('total_bill', axis=1)
y=df['total_bill']

In [126]:
x

,tip,sex,smoker,day,time,size
0,1.01,Female,No,Sun,0,2
1,1.66,Male,No,Sun,0,3
2,3.50,Male,No,Sun,0,3
3,3.31,Male,No,Sun,0,2
4,3.61,Female,No,Sun,0,4
...,...,...,...,...,...,...
239,5.92,Male,No,Sat,0,3
240,2.00,Female,Yes,Sat,0,2
241,2.00,Male,Yes,Sat,0,2
242,1.75,Male,No,Sat,0,2


In [127]:
y

0      16.99
1      10.34
2      21.01
3      23.68
4      24.59
       ...  
239    29.03
240    27.18
241    22.67
242    17.82
243    18.78
Name: total_bill, Length: 244, dtype: float64

In [128]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.20,random_state=1)

In [129]:
x_train.shape, x_test.shape

((195, 6), (49, 6))

In [130]:
# handling the missing value
# data encoding
# feature scaling
from sklearn.impute import SimpleImputer # For missing value
from sklearn.preprocessing import OneHotEncoder  #forencodinger 
from sklearn.preprocessing import StandardScaler # For scaling

from sklearn.pipeline import Pipeline  # A sequence of data transformer 
from sklearn.compose import ColumnTransformer # groups all the pipeline steps for each of the columns

In [131]:
cat_cols=['sex','smoker','day','time']
nums_cols=['tip','size']

In [132]:
num_pipeline=Pipeline(steps=[('imputation',SimpleImputer(strategy='median')),
               ('scaling',StandardScaler())])

cat_pipeline=Pipeline(steps=[('imputation',SimpleImputer(strategy='most_frequent')),
                            ('encoding',OneHotEncoder())])

In [133]:
preprocessor=ColumnTransformer([('num_pipeline',num_pipeline, nums_cols),
                  ('cat_pipeline',cat_pipeline, cat_cols)])
preprocessor

ColumnTransformer(transformers=[('num_pipeline',
                                 Pipeline(steps=[('imputation',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaling',
                                                  StandardScaler())]),
                                 ['tip', 'size']),
                                ('cat_pipeline',
                                 Pipeline(steps=[('imputation',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoding',
                                                  OneHotEncoder())]),
                                 ['sex', 'smoker', 'day', 'time'])])

In [134]:
x_train=preprocessor.fit_transform(x_train)
x_test=preprocessor.transform(x_test)

In [135]:
x_train

array([[-1.47443803, -0.57766863,  1.        , ...,  0.        ,
         1.        ,  0.        ],
       [-0.71612531,  1.47042924,  0.        , ...,  0.        ,
         1.        ,  0.        ],
       [ 1.19880579,  1.47042924,  0.        , ...,  0.        ,
         1.        ,  0.        ],
       ...,
       [ 0.43283335, -0.57766863,  1.        , ...,  1.        ,
         0.        ,  1.        ],
       [-1.29060464, -0.57766863,  0.        , ...,  0.        ,
         1.        ,  0.        ],
       [ 0.1034652 ,  0.44638031,  1.        , ...,  0.        ,
         1.        ,  0.        ]])

In [136]:
x_test

array([[-1.48209775, -1.60171757,  1.        ,  0.        ,  0.        ,
         1.        ,  0.        ,  1.        ,  0.        ,  0.        ,
         1.        ,  0.        ],
       [ 0.04984713, -0.57766863,  1.        ,  0.        ,  1.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  1.        ,
         1.        ,  0.        ],
       [ 0.36389583,  0.44638031,  0.        ,  1.        ,  0.        ,
         1.        ,  0.        ,  1.        ,  0.        ,  0.        ,
         1.        ,  0.        ],
       [-0.33313909, -0.57766863,  0.        ,  1.        ,  1.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  1.        ,
         0.        ,  1.        ],
       [ 0.04984713, -0.57766863,  0.        ,  1.        ,  1.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  1.        ,
         0.        ,  1.        ],
       [-1.14506988, -0.57766863,  0.        ,  1.        ,  0.        ,
         1.        ,  

In [138]:
# Now we build regression model
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

In [139]:
models={'support vector Regressor':SVR(),
       'DT Regressor':DecisionTreeRegressor(),
       'Multiple Linear Regression':LinearRegression(),
       'Random_forest_regressor':RandomForestRegressor()}
models

{'support vector Regressor': SVR(),
 'DT Regressor': DecisionTreeRegressor(),
 'Multiple Linear Regression': LinearRegression(),
 'Random_forest_regressor': RandomForestRegressor()}

In [141]:
from sklearn.metrics import accuracy_score

def model_train_eval(x_train,x_test, y_train, y_test, models):
    evaluation={}
    for i in range (len(models)):
        model=list(models.values())[i]
        model.fit(x_train, y_train)
        y_pred=model.predict(x_test)
        model_score=accuracy_score(y_test, y_pred)
        evaluation[list(models.keys())[i]]=model_score
    return evaluation

In [142]:
model_train_eval(x_train,x_test,y_train,y_test,models)

ValueError: continuous is not supported

# -> OOB Score (Out Of Bag)

In [115]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification

In [116]:
x,y=make_classification(n_samples=1000, n_features=20, n_classes=2, random_state=1)

In [117]:
x

array([[-2.04582165, -0.13791624, -0.08071423, ...,  2.48194524,
         0.74236675,  0.23154789],
       [-0.98726024,  1.30120189,  2.37734888, ...,  0.55445754,
        -0.21892143, -0.37608578],
       [ 0.57335921,  0.09375582,  0.4662521 , ..., -0.6088508 ,
         0.79903499, -0.17121177],
       ...,
       [-0.70737159,  1.07650943,  0.58510456, ..., -1.51337602,
         0.90239871, -0.69230951],
       [-0.20706849,  1.17319848, -1.94478665, ..., -0.32820676,
         1.5711921 ,  1.14877729],
       [-2.16769231, -2.54871672,  2.89359255, ...,  0.71535366,
         0.34329241,  1.07350284]])

In [118]:
y

array([0, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1,
       1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0,
       0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1,
       0, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1,
       1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 0,
       0, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0, 0, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0,
       1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1,
       0, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0,
       0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0,
       0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1,
       0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 0,
       1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 1, 0,
       0, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0,
       0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [119]:
rf_clf=RandomForestClassifier(n_estimators=100, oob_score=True, random_state=1)
rf_clf

RandomForestClassifier(oob_score=True, random_state=1)

In [120]:
rf_clf.fit(x,y)

RandomForestClassifier(oob_score=True, random_state=1)

In [121]:
rf_clf.oob_score_

0.861